# Big Data for Engineers 2020 - Cheat Sheet

During the exam, you will be required to write SQL, Spark RDD, and SparkSQL queries using a Jupyter notebook.

This notebook is designed to help you start writing your queries by providing you an environment with the datasets loaded and a simple query that you can use to recap the syntax of each query language.

Feel free to extend this notebook and use it for preparing the answers you need for the exam. Take into attention that the content of this notebook will not be considered for grading.

## SQL

There is a local PostgreSQL 12 installation with a dataset loaded into a database. Run the next cell to connect to it.

In [ ]:
%load_ext sql
%sql postgresql://postgres:example@db
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

To print the tables currently loaded in the database run:

In [ ]:
%%sql

SELECT * 
FROM INFORMATION_SCHEMA.TABLES 
WHERE TABLE_TYPE = 'BASE TABLE' and TABLE_CATALOG = 'postgres' and TABLE_SCHEMA = 'public';

To print the attributes of a particular table ('companies', for example) run:

In [ ]:
%%sql

SELECT column_name, data_type, character_maximum_length
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE table_name = 'companies';

In PostgreSQL use 'limit' to limit the number of returned entries. Note that in MSSQL you used top(1) instead of 'limit 1'. For example, this is a valid PostgreSQL query to select one output entry:

In [ ]:
%%sql

SELECT * 
FROM companies 
LIMIT 1;

A more complex PostgreSQL query would look like:

In [ ]:
%%sql
SELECT j.Salary, COUNT(*) AS count
FROM employees AS e
JOIN jobs AS j ON j.Id = e.Job_Id
JOIN (SELECT * FROM companies) AS c ON c.iD = j.Company_Id
WHERE c.Name LIKE 'Microsoft'
GROUP BY j.Salary
HAVING j.Salary > 100
ORDER BY count ASC
LIMIT 1

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now its your turn, you can write all your queries in new cells below. Feel free to add as many cells as needed.

## Spark RDD

The following cells show you how to load and play with a the local JSON dataset using Spark RDDs. To load Spark run the following cell:

In [ ]:
import json

from pyspark import SparkContext
from jsoniq import RumbleSession

spark = RumbleSession \
    .builder \
    .appName("Big Data examination") \
    .getOrCreate()

sc = spark.sparkContext

To load the JSON data used for the exam and load it into an array of JSON objects, run:

In [ ]:
data = sc.textFile('file:///c:\\Exam\\Files\\json_dataset\\movies.json')
entries = data.map(json.loads)

To pick one entry of the dataset use the following cell. All entries in the dataset have the same structure and so, you can use this cell to understand the structure of the data.

In [ ]:
entries.take(1)

A more complex Spark RDD query would look like:

In [ ]:
output = entries.flatMap(lambda e: e['actors'])
output = output.filter(lambda e : int(e['birth']) > 1920)
output = output.map(lambda e : (int(e['death']) - int(e['birth']), e['name']))
output = output.reduceByKey(lambda e1, e2: 
                            len(e1) if isinstance(e1, str) else e1 + 
                            len(e2) if isinstance(e2, str) else e2)
output = output.sortBy(lambda e: e, ascending=False)
print(output.count())
output.take(10)

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now its your turn, you can write all your queries in new cells below. Feel free to add as many cells as needed.

## Spark SQL

The following cells show you how to load and play with a the local JSON dataset using Spark SQL. To load Spark SQL and create a new session run the following cells:

In [ ]:
%load_ext sparksql_magic

movies_df = spark.read.json('movies.jsonl')
movies_df.createOrReplaceTempView("movies")

To print the schema of the dataset, run the following cell. For simplicity, the provided dataset does not contain missing fields or null values.

In [ ]:
movies_df.printSchema()

To pick one entry of the dataset use the following cell run:

In [ ]:
%%sparksql
SELECT *
FROM movies 
LIMIT 1

A more complex SparkSQL query would look like (do not run this code):

In [ ]:
%%sparksql
SELECT *
FROM (

    SELECT year, flat_actors.birth, flat_actors.death, flat_actors.name, (flat_actors.death - flat_actors.birth) as delta
    FROM movies LATERAL VIEW explode(actors) as flat_actors
)
WHERE delta = 100 
ORDER BY year desc, name
LIMIT 1

##### Note: the examples provided above do not contain all the query operations you might need during the exam.

Now its your turn, you can write all your queries in new cells below. Feel free to add as many cells as needed.